In [1]:
import pandas as pd
import os
import glob
from datetime import datetime
import openpyxl  # For reading existing Excel files
from openpyxl.styles import Font, PatternFill
import xlsxwriter  # For writing new Excel files with formatting
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule

# Function to load the CSV file based on prefix (e.g., 'teal')
def find_csv_file_by_prefix(prefix):
    current_dir = os.getcwd()  # Get current working directory
    file_pattern = os.path.join(current_dir, f"{prefix}*.csv")  # Search for CSV files starting with the prefix
    files = glob.glob(file_pattern)
    if files:
        return files[0]  # Return the first matching file
    else:
        return None

def load_vm_analysis_file():
    current_dir = os.getcwd()  # Get current working directory
    file_pattern = os.path.join(current_dir, '*VM Analysis*.xlsx')  # Pattern to match VM Analysis file
    files = glob.glob(file_pattern)

    if files:
        vm_analysis_file = files[0]
        print(f"Found VM Analysis file: {vm_analysis_file}")

        # Load the workbook in normal mode to preserve conditional formatting
        workbook = load_workbook(vm_analysis_file, read_only=False)

        # Load the data from the sheets into Pandas DataFrames
        spend_summary = pd.read_excel(vm_analysis_file, sheet_name='Spend Summary', engine='openpyxl')
        summary = pd.read_excel(vm_analysis_file, sheet_name='Summary', engine='openpyxl')

        print(f"Loaded Spend Summary and Summary sheets from {vm_analysis_file}")
        return spend_summary, summary, workbook  # Return workbook as third value
    else:
        print("No VM Analysis file found.")
        return None, None, None

# Load the NAICS Library file using openpyxl
def load_naics_library_file():
    current_dir = os.getcwd()
    file_pattern = os.path.join(current_dir, '*NAICS Library*.xlsx')
    files = glob.glob(file_pattern)
    if files:
        naics_library_file = files[0]
        naics_library_df = pd.read_excel(naics_library_file, engine='openpyxl')
        print(f"Loaded NAICS Library from {naics_library_file}")
        return naics_library_df
    else:
        print("No NAICS Library file found.")
        return None

# Function to apply conditional formatting to the Excel file
def add_conditional_formatting_to_output(file_name):
    # Load the generated Excel file using openpyxl
    workbook = load_workbook(file_name)

    # Access the worksheet where you want to apply conditional formatting
    worksheet = workbook['NAICS Pivot']  # Adjust based on where you want the formatting

    # Define a conditional formatting rule (e.g., 2-color scale from green to red)
    color_scale_rule = ColorScaleRule(start_type="min", start_color="FF00FF00",  # Green for min value
                                      end_type="max", end_color="FFFF0000")     # Red for max value

    # Apply the rule to a range of cells (adjust the range as needed)
    worksheet.conditional_formatting.add("B2:B1000", color_scale_rule)  # Example: Apply to Supplier Count
    worksheet.conditional_formatting.add("D2:D1000", color_scale_rule)  # Example: Apply to Supplier Spend

    # Save the workbook with the conditional formatting applied
    workbook.save(file_name)  
    
# Custom function to combine and sort NAICS codes
def combine_and_sort_naics(naics_list):
    def custom_sort_key(code):
        # Sort by first two characters, then by length
        if code.startswith(('31', '32', '33')):
            return ('31-33', len(code), code)
        elif code.startswith(('44', '45')):
            return ('44-45', len(code), code)
        elif code.startswith(('48', '49')):
            return ('48-49', len(code), code)
        return (code[:2], len(code), code)
    
    # Create a unique NAICS list and sort it
    unique_naics = sorted(set(naics_list), key=custom_sort_key)
    
    # Ensure "31-33", "44-45", and "48-49" are included only once
    if any(code.startswith(('31', '32', '33')) for code in naics_list):
        unique_naics = ['31-33'] + [code for code in unique_naics if not code.startswith(('31', '32', '33'))]
    if any(code.startswith(('44', '45')) for code in naics_list):
        unique_naics = ['44-45'] + [code for code in unique_naics if not code.startswith(('44', '45'))]
    if any(code.startswith(('48', '49')) for code in naics_list):
        unique_naics = ['48-49'] + [code for code in unique_naics if not code.startswith(('48', '49'))]
    
    return unique_naics

# Function to ensure both NAICS Code columns are formatted consistently (as strings, padded if needed)
def format_naics_codes(naics_code):
    # Ensure NAICS codes are strings and pad with zeros if necessary
    naics_code_str = str(naics_code).strip()  # Convert to string and strip whitespace
    return naics_code_str.zfill(6)  # Pad with leading zeros to ensure 6-character format

# Replace NAICS Codes with values from the NAICS Library file
def replace_naics_codes_with_library(naics_pivot_df, naics_library_df):
    # Ensure both NAICS Code columns are formatted consistently
    naics_pivot_df['NAICS Code'] = naics_pivot_df['NAICS Code'].apply(format_naics_codes)
    naics_library_df['2022 NAICS Code'] = naics_library_df['2022 NAICS Code'].apply(format_naics_codes)
    
    # Create a mapping dictionary from the NAICS Library file
    naics_mapping = naics_library_df.set_index('2022 NAICS Code')['NAICS Concat'].to_dict()

    # Replace only values that are found in the NAICS mapping dictionary
    naics_pivot_df['NAICS Code'] = naics_pivot_df['NAICS Code'].apply(
        lambda x: naics_mapping.get(x, x)  # Keep original value if not found in mapping
    )
    
    return naics_pivot_df


# Function to generate the NAICS List tab
def generate_naics_list_tab(df, spend_df):
    naics_list = []

    # Filter rows where enriched_field is 'naics'
    naics_rows = df[df['enriched_field'] == 'naics']
    
    # Go through the filtered rows and extract NAICS codes
    for _, row in naics_rows.iterrows():
        supplier_id = row['internal_supplier_id_or_vendor_number']
        
        # Extract all merged_value columns (merged_value_1 through merged_value_10)
        merged_values = row[['merged_value_1', 'merged_value_2', 'merged_value_3', 'merged_value_4',
                             'merged_value_5', 'merged_value_6', 'merged_value_7', 'merged_value_8', 
                             'merged_value_9', 'merged_value_10']].values
        
        # Add non-empty NAICS values to the list along with Supplier ID
        for naics_code in merged_values:
            if pd.notnull(naics_code) and naics_code != '':
                naics_list.append({
                    'NAICS': str(naics_code),
                    'Internal Supplier ID': supplier_id,
                    'Supplier Name': extract_value_by_field(df, supplier_id, 'company_name'),
                    'Aggregated Spend': extract_aggregated_spend(supplier_id, spend_df)
                })

    naics_list_df = pd.DataFrame(naics_list)

    # Count occurrences of each NAICS code and add as a new column
    naics_list_df['NAICS Count'] = naics_list_df.groupby('NAICS')['NAICS'].transform('count')

    # Sort by NAICS Count in descending order
    naics_list_df = naics_list_df.sort_values(by='NAICS Count', ascending=False).reset_index(drop=True)

    return naics_list_df

# Function to generate the NAICS Codes tab (existing logic)
def generate_naics_codes_tab(df, supplier_ids, spend_df):
    naics_report = []
    
    for supplier_id in supplier_ids:
        supplier_name = extract_value_by_field(df, supplier_id, 'company_name')
        supplier_address = extract_value_by_field(df, supplier_id, 'complete_address')
        aggregated_spend = extract_aggregated_spend(supplier_id, spend_df)

        naics_2_digit = get_naics_code(supplier_id, df, 2)
        naics_3_digit = get_naics_code(supplier_id, df, 3)
        naics_4_digit = get_naics_code(supplier_id, df, 4)
        naics_5_digit = get_naics_code(supplier_id, df, 5)
        naics_6_digit = get_naics_code(supplier_id, df, 6)
        
        naics_report.append({
            'Internal Supplier ID': supplier_id,
            'Supplier Name': supplier_name,
            'Supplier Address': supplier_address,
            'Aggregated Spend': aggregated_spend,
            '2-digit NAICS': naics_2_digit,
            '3-digit NAICS': naics_3_digit,
            '4-digit NAICS': naics_4_digit,
            '5-digit NAICS': naics_5_digit,
            '6-digit NAICS': naics_6_digit,
        })

    return pd.DataFrame(naics_report)

# Function for generating NAICS Pivot tab with exact match logic
def generate_naics_pivot_tab(naics_list_df, spend_df, summary_df):
    # Get the unique NAICS codes and apply custom sorting logic
    naics_codes = combine_and_sort_naics(naics_list_df['NAICS'].tolist())
    
    pivot_data = []
    
    # Get the total supplier count and total spend from the Summary tab in VM Analysis file
    total_suppliers = summary_df.loc[summary_df['Label'] == 'GRAND TOTAL', 'Count'].values[0]
    total_spend = spend_df['aggregated_spend'].sum()

    # Populate the pivot data based on the new Supplier Count logic (exact match)
    for naics_code in naics_codes:
        if naics_code == '31-33':
            supplier_count = (
                naics_list_df[naics_list_df['NAICS'] == '31'].shape[0] +
                naics_list_df[naics_list_df['NAICS'] == '32'].shape[0] +
                naics_list_df[naics_list_df['NAICS'] == '33'].shape[0]
            )
            supplier_spend = naics_list_df[naics_list_df['NAICS'].isin(['31', '32', '33'])]['Aggregated Spend'].sum()
        elif naics_code == '44-45':
            supplier_count = (
                naics_list_df[naics_list_df['NAICS'] == '44'].shape[0] +
                naics_list_df[naics_list_df['NAICS'] == '45'].shape[0]
            )
            supplier_spend = naics_list_df[naics_list_df['NAICS'].isin(['44', '45'])]['Aggregated Spend'].sum()
        elif naics_code == '48-49':
            supplier_count = (
                naics_list_df[naics_list_df['NAICS'] == '48'].shape[0] +
                naics_list_df[naics_list_df['NAICS'] == '49'].shape[0]
            )
            supplier_spend = naics_list_df[naics_list_df['NAICS'].isin(['48', '49'])]['Aggregated Spend'].sum()
        else:
            supplier_count = naics_list_df[naics_list_df['NAICS'] == naics_code].shape[0]
            supplier_spend = naics_list_df[naics_list_df['NAICS'] == naics_code]['Aggregated Spend'].sum()
        
        percentage_suppliers = (supplier_count / total_suppliers) if total_suppliers > 0 else 0
        percentage_spend = (supplier_spend / total_spend) if total_spend > 0 else 0
        
        pivot_data.append({
            'NAICS Code': naics_code,
            'Supplier Count': supplier_count,
            'Percentage of Suppliers': round(percentage_suppliers, 2),
            'Supplier Spend': round(supplier_spend, 2),
            'Percentage of Spend': round(percentage_spend, 2)
        })
    
    return pd.DataFrame(pivot_data)

# Helper functions used in generating tabs
def extract_value_by_field(df, supplier_id, field_name):
    filtered_rows = df[(df['internal_supplier_id_or_vendor_number'] == supplier_id) & (df['enriched_field'] == field_name)]
    if not filtered_rows.empty:
        value = filtered_rows['merged_value_1'].dropna().values
        if len(value) > 0:
            return value[0]
    return None

def extract_aggregated_spend(supplier_id, spend_df):
    # Ensure the column names are correct
    spend_value = spend_df.loc[spend_df['internal_supplier_id'] == supplier_id, 'aggregated_spend']
    
    if not spend_value.empty:
        return round(spend_value.values[0], 2)
    return 0

def get_naics_code(supplier_id, df, length):
    naics_rows = df[(df['internal_supplier_id_or_vendor_number'] == supplier_id) & (df['enriched_field'] == 'naics')]
    merged_values = naics_rows[[
        'merged_value_1', 'merged_value_2', 'merged_value_3', 'merged_value_4', 
        'merged_value_5', 'merged_value_6', 'merged_value_7', 'merged_value_8', 
        'merged_value_9', 'merged_value_10'
    ]].values.flatten()
    naics_codes = [str(val) for val in merged_values if pd.notnull(val) and len(str(val)) == length]
    return '|'.join(naics_codes)

# Apply number and percentage formatting to specific columns
def apply_formatting(writer, worksheet, format_type, col_range):
    if format_type == 'number':
        num_format = writer.book.add_format({'num_format': '#,##0.00'})
    elif format_type == 'percentage':
        num_format = writer.book.add_format({'num_format': '0.00%'})
    
    for col in col_range:
        worksheet.set_column(f'{col}:{col}', None, num_format)

# Apply conditional formatting for two-color scale (orange to white)
def apply_conditional_formatting(worksheet, col_range):
    for col in col_range:
        worksheet.conditional_format(f'{col}2:{col}1000', {'type': '2_color_scale',
                                                           'min_color': '#FFFFFF',
                                                           'max_color': '#FFA500'})

# Apply teal highlight, white font, and bold formatting to headers
def apply_header_formatting(writer, worksheet, df):
    header_format = writer.book.add_format({
        'bold': True,
        'bg_color': '#008080',  # Teal background
        'font_color': 'white',  # White font color
        'border': 1
    })
    # Apply the header format to the first row
    for col_num, column_name in enumerate(df.columns):
        worksheet.write(0, col_num, column_name, header_format)

# Apply light teal highlight to Aggregated Spend column
def apply_aggregated_spend_highlight(writer, worksheet, col_letter):
    highlight_format = writer.book.add_format({
        'bg_color': '#CCFFFF'  # Light teal background
    })
    worksheet.conditional_format(f'{col_letter}2:{col_letter}1000', {'type': 'cell',
                                                                     'criteria': '>=',
                                                                     'value': 0,
                                                                     'format': highlight_format})

def main():
    # Load the teal CSV file, VM Analysis Excel file, and NAICS Library file
    csv_file = find_csv_file_by_prefix('teal')
    spend_df, summary_df, workbook = load_vm_analysis_file()
    naics_library_df = load_naics_library_file()

    # Check if files were loaded successfully
    if csv_file and spend_df is not None and naics_library_df is not None:
        print(f"Processing CSV file: {csv_file}")
        df = pd.read_csv(csv_file, low_memory=False)

        # Standardize column names to avoid case or space issues
        spend_df.columns = spend_df.columns.str.strip().str.lower().str.replace(' ', '_')

        # Print columns for debugging (this will help us see the actual columns)
        print("Spend DataFrame columns:", spend_df.columns)

        # Extract unique supplier IDs from the CSV
        supplier_ids = df['internal_supplier_id_or_vendor_number'].unique()

        # Generate NAICS List, NAICS Codes, and NAICS Pivot tabs
        naics_list_df = generate_naics_list_tab(df, spend_df)
        naics_codes_df = generate_naics_codes_tab(df, supplier_ids, spend_df)
        naics_pivot_df = generate_naics_pivot_tab(naics_list_df, spend_df, summary_df)

        # Replace NAICS Codes with values from the NAICS Library file
        naics_pivot_df = replace_naics_codes_with_library(naics_pivot_df, naics_library_df)

        # Add datestamp to the output file name
        current_date = datetime.now().strftime('%Y-%m-%d')
        output_file = f'Step 3_SDP NAICS Refined - {current_date}.xlsx'

        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            # Write NAICS List sheet
            naics_list_df.to_excel(writer, sheet_name='NAICS List', index=False)
            worksheet1 = writer.sheets['NAICS List']
            apply_header_formatting(writer, worksheet1, naics_list_df)

            worksheet1.set_column('A:A', 15)  # NAICS
            worksheet1.set_column('B:B', 20)  # Internal Supplier ID
            worksheet1.set_column('C:C', 50)  # Supplier Name
            worksheet1.set_column('D:D', 20)  # Aggregated Spend
            worksheet1.set_column('E:E', 12)  # NAICS Count

            # Apply number formatting to Aggregated Spend
            apply_formatting(writer, worksheet1, 'number', ['D'])

            # Apply light teal highlight to Aggregated Spend
            apply_aggregated_spend_highlight(writer, worksheet1, 'D')

            # Write NAICS Codes sheet
            naics_codes_df.to_excel(writer, sheet_name='NAICS Codes', index=False)
            worksheet2 = writer.sheets['NAICS Codes']
            apply_header_formatting(writer, worksheet2, naics_codes_df)

            worksheet2.set_column('A:A', 20)  # Internal Supplier ID
            worksheet2.set_column('B:B', 50)  # Supplier Name
            worksheet2.set_column('C:C', 50)  # Supplier Address
            worksheet2.set_column('D:D', 20)  # Aggregated Spend
            worksheet2.set_column('E:E', 12)  # 2-digit NAICS
            worksheet2.set_column('F:F', 12)  # 3-digit NAICS
            worksheet2.set_column('G:G', 12)  # 4-digit NAICS
            worksheet2.set_column('H:H', 12)  # 5-digit NAICS
            worksheet2.set_column('I:I', 12)  # 6-digit NAICS

            # Apply number formatting to Aggregated Spend
            apply_formatting(writer, worksheet2, 'number', ['D'])

            # Apply light teal highlight to Aggregated Spend
            apply_aggregated_spend_highlight(writer, worksheet2, 'D')

            # Write NAICS Pivot sheet
            naics_pivot_df.to_excel(writer, sheet_name='NAICS Pivot', index=False)
            worksheet3 = writer.sheets['NAICS Pivot']
            apply_header_formatting(writer, worksheet3, naics_pivot_df)

            worksheet3.set_column('A:A', 15)  # NAICS Code
            worksheet3.set_column('B:B', 20)  # Supplier Count
            worksheet3.set_column('C:C', 20)  # Percentage of Suppliers
            worksheet3.set_column('D:D', 20)  # Supplier Spend
            worksheet3.set_column('E:E', 20)  # Percentage of Spend

            # Apply number formatting to Supplier Spend
            apply_formatting(writer, worksheet3, 'number', ['D'])

            # Apply percentage formatting to Percentage of Suppliers and Percentage of Spend
            apply_formatting(writer, worksheet3, 'percentage', ['C', 'E'])

            # Apply conditional formatting (two-color scale) to relevant columns
            apply_conditional_formatting(worksheet3, ['B', 'C', 'D', 'E'])

        # Reopen the file with openpyxl to apply conditional formatting
        add_conditional_formatting_to_output(output_file)

        print(f"Excel file '{output_file}' has been created with conditional formatting.")

    else:
        print("Error: Missing required files.")

                 
if __name__ == "__main__":
    main()



Found VM Analysis file: /Users/louis.standridge/Desktop/Macys SDP Reporting Package/macys - SDP - VM Analysis - 2024-10-11.xlsx
Loaded Spend Summary and Summary sheets from /Users/louis.standridge/Desktop/Macys SDP Reporting Package/macys - SDP - VM Analysis - 2024-10-11.xlsx
Loaded NAICS Library from /Users/louis.standridge/Desktop/Macys SDP Reporting Package/Step 3_SDP NAICS Library.xlsx
Processing CSV file: /Users/louis.standridge/Desktop/Macys SDP Reporting Package/teal_iq - Macys - 10.11.2024.csv


/Users/louis.standridge/opt/anaconda3/lib/python3.8/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


Spend DataFrame columns: Index(['internal_supplier_id', 'aggregated_spend', 'spend_percentage'], dtype='object')
Excel file 'Step 3_SDP NAICS Refined - 2024-10-18.xlsx' has been created with conditional formatting.
